<a href="https://colab.research.google.com/github/ValentinaEmili/Texture-synthesis/blob/main/generation/new_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch-fidelity
!pip install import-ipynb -q

import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import time
import numpy as np
import math
import sys
from torch_fidelity import calculate_metrics
import import_ipynb

REPO_DIR = '/content/Texture_synthesis'
os.chdir('/content')

if not os.path.exists(REPO_DIR):
  !git clone https://github.com/ValentinaEmili/Texture-synthesis.git Texture_synthesis

if REPO_DIR not in sys.path:
  sys.path.append(REPO_DIR)

In [3]:
model_type = 'vq_vae'
#model_type = 'vq_gan'
#model_type = 'vq_gan_tt'

#split = '1'
#split = '2'
split = '1_2'

num_embeddings=512
#num_embeddings=256

img_size=512
#img_size=256

train_file = 'train_' + split + '.txt'
val_file = 'val_' + split + '.txt'
test_file = 'test_' + split + '.txt'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8

from Texture_synthesis.generation.Transformers import Transformer

if model_type == 'vq_vae':
  from Texture_synthesis.codebook.VQ_VAE import VQ_VAE
  generator = VQ_VAE(hidden_dim=128, embedding_dim=128, num_embeddings=num_embeddings).to(device)
  seq_len, num_embeddings_T = 1024, 512
  quant_save_path = f'/content/drive/MyDrive/DeepLearning/dtd/new_checkpoints/{model_type}/best_{split}_128_{num_embeddings}.pth'

elif model_type == 'vq_gan':
  from Texture_synthesis.codebook.VQGAN import VQGAN, Discriminator
  generator = VQGAN(hidden_dim=128, embedding_dim=128, num_embeddings=num_embeddings).to(device)
  seq_len, num_embeddings_T = 1024, 512
  quant_save_path = f'/content/drive/MyDrive/DeepLearning/dtd/new_checkpoints/{model_type}/gen_best_{split}_128_{num_embeddings}.pth'

else:
  from Texture_synthesis.codebook.VQGAN_TT import VQGAN, Discriminator
  generator = VQGAN().to(device)
  #seq_len, num_embeddings_T = 256, 1024
  quant_save_path = f'/content/drive/MyDrive/DeepLearning/dtd/new_checkpoints/vq_gan/taming_transformers/gen_best_{split}.pth'

checkpoint = torch.load(quant_save_path, weights_only=True)
generator.load_state_dict(checkpoint['model_state_dict'])

best_save_path = f'/content/drive/MyDrive/DeepLearning/dtd/new_checkpoints/transformers/split_{split}/{img_size}/{model_type}_{num_embeddings}.pth'
transformer = Transformer(num_embeddings=num_embeddings_T, seq_len=seq_len, n_layers=8).to(device)
optimizer = optim.Adam(transformer.parameters(), lr=2e-4, betas=(0.9, 0.999))

out_dir = f'/content/drive/MyDrive/DeepLearning/dtd/generated_images/split_{split}/{model_type}'
real_dir = f"drive/MyDrive/DeepLearning/dtd/test_images/split_{split}"
recon_save_dir = f'/content/drive/MyDrive/DeepLearning/dtd/transformer_reconstructed_images/split_{split}/{model_type}'

In [6]:
class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [7]:
train_transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.RandomCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, train_file), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, val_file), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, test_file), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [8]:
@torch.no_grad()
def extract_codebook_indices(quantizer, data):
  quantizer.eval()

  embedding_dim = quantizer.vq.embedding_dim
  embeddings = quantizer.vq.embeddings.weight                               # (num_embeddings, embed_dim)

  z = quantizer.encoder(data)                                               # (batch_size, embed_dim, h, w)
  z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)  # (batch_size * h * w, embed_dim)

  distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))        # (batch_size * h * w, num_embeddings)

  indices = torch.argmin(distances, dim=1)        # (batch_size * h * w)
  return indices.reshape(z.shape[0],-1)           # (batch_size, seq_len)

# Validation

In [9]:
@torch.no_grad()
def validate(val_loader, transformer, quantizer, device):
  transformer.eval()
  quantizer.eval()

  transformer.to(device)
  quantizer.to(device)

  bos_token_id = transformer.num_embeddings

  total_loss, total_accuracy, total_tokens = 0.0, 0.0, 0.0

  for batch_idx, (data, _) in enumerate(val_loader):
    data = data.to(device)

    indices = extract_codebook_indices(quantizer, data)
    b, t = indices.shape                      # (batch_size, seq_len)

    bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
    full_sequence = torch.cat([bos_tokens, indices], dim=1) # (batch_size, seq_len)

    inputs = full_sequence[:, :-1]            # [BOS, c1, c2, ..., c_{t-1}]
    targets = full_sequence[:, 1:]            # [c1, c2, ..., c_{t-1}, c_{t}]

    num_tokens = targets.numel()
    total_tokens += num_tokens

    logits = transformer(inputs)              # (batch_size, seq_len, num_embeddings)

    predictions = torch.argmax(logits, dim=-1)
    accuracy = (predictions == targets).sum().item()

    targets = targets.reshape(-1)                             # (batch_size * seq_len)
    logits = logits.reshape(-1, transformer.num_embeddings) # (batch_size * seq_len, num_embeddings)

    loss = F.cross_entropy(logits, targets, reduction="sum")

    total_loss += loss.item()
    total_accuracy += accuracy

  mean_loss = total_loss / total_tokens
  mean_accuracy = (total_accuracy / total_tokens) * 100
  return mean_loss, mean_accuracy

# Training

In [10]:
def train_and_validation(train_loader, val_loader, transformer, quantizer, optimizer, device, best_val_loss=float('inf'), epochs=50, next_epoch=0):
  transformer.to(device)
  quantizer.to(device)

  bos_token_id = transformer.num_embeddings

  for epoch in range(next_epoch, epochs):
    transformer.train()
    quantizer.eval()
    total_loss, total_accuracy, total_tokens = 0.0, 0.0, 0

    for batch_idx, (data, _) in enumerate(train_loader):
      data = data.to(device)

      indices = extract_codebook_indices(quantizer, data)
      b, t = indices.shape                      # (batch_size, seq_len)

      bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
      full_sequence = torch.cat([bos_tokens, indices], dim=1) # (batch_size, seq_len)

      inputs = full_sequence[:, :-1]            # [BOS, c1, c2, ..., c_{t-1}]
      targets = full_sequence[:, 1:]            # [c1, c2, ..., c_{t-1}, c_{t}]

      num_tokens = targets.numel()
      total_tokens += num_tokens

      logits = transformer(inputs)              # (batch_size, seq_len, num_embeddings)

      predictions = torch.argmax(logits, dim=-1)
      accuracy = (predictions == targets).sum().item()

      optimizer.zero_grad()

      targets = targets.reshape(-1)                             # (batch_size * seq_len)
      logits = logits.reshape(-1, transformer.num_embeddings) # (batch_size * seq_len, num_embeddings)

      loss = F.cross_entropy(logits, targets)

      loss.backward()
      optimizer.step()

      total_loss += loss.item() * num_tokens
      total_accuracy += accuracy

    train_loss = total_loss / total_tokens
    train_accuracy = (total_accuracy / total_tokens) * 100
    train_perplexity = np.exp(train_loss)

    print(f"====> Epoch {epoch} Finished")
    print(f"====> Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f} | Train Perplexity: {train_perplexity:.2f}")

    val_loss, val_accuracy = validate(val_loader, transformer, quantizer, device)
    val_perplexity = np.exp(val_loss)
    print(f"====> Valid Loss: {val_loss:.4f} | Valid Accuracy: {val_accuracy:.4f} | Valid Perplexity: {val_perplexity:.2f}")

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      torch.save({
          'epoch': epoch,
          'model_state_dict':transformer.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(),
          'best_val_loss': best_val_loss,
          'perplexity': val_perplexity,
          'accuracy': val_accuracy
          }, best_save_path)
      print(f"New best model at epoch {epoch}\n")

In [ ]:
train_and_validation(train_loader, val_loader, transformer, generator, optimizer, device, epochs=50)

In [ ]:
checkpoint = torch.load(best_save_path, weights_only=False)
next_epoch = checkpoint['epoch'] + 1
transformer.load_state_dict(checkpoint['model_state_dict'])
best_val_loss = checkpoint['best_val_loss']

train_and_validation(train_loader, val_loader, transformer, generator, optimizer, device, epochs=50, best_val_loss=best_val_loss, next_epoch=next_epoch)